# Scraping

In [2]:
import pandas as pd
from selenium.webdriver.common.by import By
from selenium import webdriver
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from utils.constants import *
from utils.scraping import *

In [3]:
TYPE = 'mengubah'
LAW = 'uu'
RELATIONSHIP_TYPE = TYPE.upper()

web = f"https://peraturan.go.id/{TYPE}/{LAW}"


In [4]:
service = Service(executable_path=CHROMEDRIVER_PATH)
driver = webdriver.Chrome(service=service)
driver.get(web)

In [5]:
# Find the table rows using XPath
rows = driver.find_elements(By.XPATH, "//table/tbody/tr")

# Initialize a set to hold unique rows and a list to collect the data
unique_rows = set()
data = []

for row in rows:
    # Extract data from each column
    no = row.find_element(By.XPATH, "./th").text
    peraturan_element = row.find_element(By.XPATH, "./td[1]/a")
    peraturan = peraturan_element.text
    peraturan_link = peraturan_element.get_attribute("href")
    status = row.find_element(By.XPATH, "./td[2]").text
    related_peraturan_elements = row.find_elements(By.XPATH, "./td[3]/a")

    # Create unique rows by iterating over related peraturan
    for related_peraturan_element in related_peraturan_elements:
        related_peraturan = related_peraturan_element.text
        related_peraturan_link = related_peraturan_element.get_attribute("href")
        
        # Create a tuple representing the row with both links
        row_tuple = (peraturan, status, related_peraturan, peraturan_link, related_peraturan_link)
        
        # Add the tuple to the set to ensure uniqueness
        if row_tuple not in unique_rows:
            unique_rows.add(row_tuple)
            data.append(row_tuple)

# Convert the list of tuples to a DataFrame
df = pd.DataFrame(data, columns=["PERATURAN", "STATUS", "RELATED_PERATURAN", "PERATURAN_LINK", "RELATED_PERATURAN_LINK"])

In [10]:
# Use df.head(x) to only check x rows first
df2 = df

In [11]:
# Define the exact key mappings
key_mapping = {
    "Jenis/Bentuk Peraturan": "jenis_peraturan",
    "Pemrakarsa": "pemrakarsa",
    "Nomor": "nomor",
    "Tahun": "tahun",
    "Tentang": "tentang",
    "Tempat Penetapan": "tempat_penetapan",
    "Status": "status",
}

# Dictionary to cache fetched metadata
metadata_cache = {}

title_div_xpath = '//div[@class="detail_title_1"]'
table_xpath = '//table[@class="table caption-top"]'
row_xpath = './/tr'
row_q_xpath = './/th[@style="width: 30%;"]'
row_a_xpath = './/td'

SKIP_COLS = [
    "pejabat_yang_menetapkan", "tahun_pengundangan", "nomor_pengundangan",
    "nomor_tambahan", "tanggal_pengundangan", "pejabat_pengundangan"
]

# Function to fetch metadata for a given link
def fetch_metadata(link):
    # Check if the link is already in the cache
    if link in metadata_cache:
        return metadata_cache[link]

    driver.get(link)
    WebDriverWait(driver, 10).until(EC.presence_of_all_elements_located((By.XPATH, table_xpath)))

    reg_detail = {}
    title_element = driver.find_element(By.XPATH, title_div_xpath)
    reg_detail["title"] = title_element.text

    tables = driver.find_elements(By.XPATH, table_xpath)

    for table in tables:
        rows = table.find_elements(By.XPATH, row_xpath)
        for row_element in rows:
            row_q = row_element.find_element(By.XPATH, row_q_xpath)
            row_q_text = row_q.text.strip()

            if row_q_text in SKIP_COLS:
                continue

            # Capture the text content properly even if there are nested elements
            row_a = row_element.find_element(By.XPATH, row_a_xpath)
            reg_detail[key_mapping.get(row_q_text, row_q_text)] = row_a.get_attribute('textContent').strip()

    # Store the fetched metadata in the cache
    metadata_cache[link] = reg_detail

    return reg_detail


# List to hold the combined metadata for each row
combined_metadata = []

for index, row in df2.iterrows():
    print(f"Processing row {index + 1}/{len(df2)}")

    # Fetch metadata for PERATURAN_LINK
    peraturan_metadata = fetch_metadata(row['PERATURAN_LINK'])

    # Fetch metadata for RELATED_PERATURAN_LINK
    related_peraturan_metadata = fetch_metadata(row['RELATED_PERATURAN_LINK'])

    # Combine both metadata into a single dictionary
    combined_metadata_row = {**row.to_dict()}
    combined_metadata_row.update({
        f"PERATURAN_{key}": peraturan_metadata.get(key, "") for key in key_mapping.values()
    })
    combined_metadata_row.update({
        f"RELATED_{key}": related_peraturan_metadata.get(key, "") for key in key_mapping.values()
    })

    combined_metadata.append(combined_metadata_row)

# Convert the list of combined metadata dictionaries to a DataFrame
df_combined = pd.DataFrame(combined_metadata)

Processing row 1/497
Processing row 2/497
Processing row 3/497
Processing row 4/497
Processing row 5/497
Processing row 6/497
Processing row 7/497
Processing row 8/497
Processing row 9/497
Processing row 10/497
Processing row 11/497
Processing row 12/497
Processing row 13/497
Processing row 14/497
Processing row 15/497
Processing row 16/497
Processing row 17/497
Processing row 18/497
Processing row 19/497
Processing row 20/497
Processing row 21/497
Processing row 22/497
Processing row 23/497
Processing row 24/497
Processing row 25/497
Processing row 26/497
Processing row 27/497
Processing row 28/497
Processing row 29/497
Processing row 30/497
Processing row 31/497
Processing row 32/497
Processing row 33/497
Processing row 34/497
Processing row 35/497
Processing row 36/497
Processing row 37/497
Processing row 38/497
Processing row 39/497
Processing row 40/497
Processing row 41/497
Processing row 42/497
Processing row 43/497
Processing row 44/497
Processing row 45/497
Processing row 46/4

In [19]:
print(len(df_combined))
df_combined.head(1)

497


,PERATURAN,STATUS,RELATED_PERATURAN,PERATURAN_LINK,RELATED_PERATURAN_LINK,PERATURAN_jenis_peraturan,PERATURAN_pemrakarsa,PERATURAN_nomor,PERATURAN_tahun,PERATURAN_tentang,PERATURAN_tempat_penetapan,PERATURAN_status,RELATED_jenis_peraturan,RELATED_pemrakarsa,RELATED_nomor,RELATED_tahun,RELATED_tentang,RELATED_tempat_penetapan,RELATED_status
0,uu-no-28-tahun-2000,Mengubah,uu-no-11-tahun-1998,https://peraturan.go.id/id/uu-no-28-tahun-2000,https://peraturan.go.id/id/uu-no-11-tahun-1998,UNDANG-UNDANG,PEMERINTAH PUSAT,28,2000,PENETAPAN PERPU 3-2000 TENTANG PERUBAHAN UU 11...,Jakarta,Tidak Berlaku Dicabut Oleh :Undang-Undang Nomo...,UNDANG-UNDANG,PEMERINTAH PUSAT,11,1998,PERUBAHAN BERLAKUNYA UU 25-1997 TENTANG KETENA...,Jakarta,Tidak Berlaku Dicabut Oleh :Undang-Undang Nomo...


In [13]:
df_combined.to_csv(f"csv/{LAW}/{TYPE}.csv", index=False)

# Inserting

In [15]:
from utils.config import get_config

config = get_config()
config

{'openai_api_Key': 'sk-proj-RU6DuTOV5RJycNxY42siT3BlbkFJrbZxugcP4wme04lJTvDj',
 'pinecone_api_key': '83aa5d86-c406-4990-b503-2f0518dce9dd',
 'cohere_api_key': '6xDdKx8a7EaYhwgOh8GCVMZi90emuG5FD754ARrh',
 'neo4j_uri': None,
 'neo4j_username': None,
 'neo4j_password': None,
 'neo4j_db': None}

In [16]:
from py2neo import Graph
from utils.config import get_config

config = get_config()

graph = Graph(config['neo4j_uri'], name=config['neo4j_username'], password=config['neo4j_password'])

In [17]:
graph

Graph('neo4j+s://0b76488f.databases.neo4j.io:7687', name='neo4j')

In [18]:
from py2neo import Node, Relationship

nodes = {}

# Function to create nodes
def create_node(label, properties):
    node_id = properties['id']
    if node_id not in nodes:
        node = Node(label, **properties)
        graph.merge(node, label, "id")
        nodes[node_id] = node
    return nodes[node_id]

# Function to create relationships
def create_relationship(source_id, target_id, relationship_type):
    source_node = nodes.get(source_id)
    target_node = nodes.get(target_id)
    if source_node and target_node:
        relationship = Relationship(source_node, relationship_type, target_node)
        graph.merge(relationship)

# Create the reverse relationship
def create_reverse_relationship(source_id, target_id, relationship_type):
    if relationship_type == "MENCABUT":
        reverse_relationship_type = "DICABUT"
    elif relationship_type == "MENGUBAH":
        reverse_relationship_type = "DIUBAH"
    else:
        raise ValueError(f"Unknown relationship type: {relationship_type}")
    
    create_relationship(target_id, source_id, reverse_relationship_type)


# Iterate over the DataFrame and create nodes and relationships
for index, row in df_combined.iterrows():
    # Create a node for PERATURAN
    peraturan_properties = {
        "id": row['PERATURAN_LINK'],
        "nomor": row['PERATURAN_nomor'],
        "jenis_peraturan": row['PERATURAN_jenis_peraturan'],
        "pemrakarsa": row['PERATURAN_pemrakarsa'],
        "tahun": row['PERATURAN_tahun'],
        "tentang": row['PERATURAN_tentang'],
        "tempat_penetapan": row['PERATURAN_tempat_penetapan'],
        "status": row['PERATURAN_status'],
    }
    peraturan_node = create_node("Peraturan", peraturan_properties)

    # Create a node for RELATED_PERATURAN
    related_properties = {
        "id": row['RELATED_PERATURAN_LINK'],
        "nomor": row['RELATED_nomor'],
        "jenis_peraturan": row['RELATED_jenis_peraturan'],
        "pemrakarsa": row['RELATED_pemrakarsa'],
        "tahun": row['RELATED_tahun'],
        "tentang": row['RELATED_tentang'],
        "tempat_penetapan": row['RELATED_tempat_penetapan'],
        "status": row['RELATED_status'],
    }
    related_node = create_node("Peraturan", related_properties)

    create_relationship(row['PERATURAN_LINK'], row['RELATED_PERATURAN_LINK'], RELATIONSHIP_TYPE)
    create_reverse_relationship(row['PERATURAN_LINK'], row['RELATED_PERATURAN_LINK'], RELATIONSHIP_TYPE)


# Langchain Graph Query

In [1]:
from utils.database.neo4j import Neo4jGraphStore
from utils.config import get_config

config = get_config()
neo4j_sikepo = Neo4jGraphStore(config=config)
graph = neo4j_sikepo.get_graph()

In [2]:
from utils.prompt.graph import GRAPH_CYPHER_GEN_PROMPT, GRAPH_QA_GEN_PROMPT
from langchain_core.prompts import PromptTemplate

qa_generation_template = GRAPH_QA_GEN_PROMPT
qa_generation_prompt = PromptTemplate(
    input_variables=["context", "question"], template=qa_generation_template)

cypher_generation_template = GRAPH_CYPHER_GEN_PROMPT
cypher_generation_prompt = PromptTemplate(input_variables=["schema", "question"], template=cypher_generation_template)

In [3]:
from langchain.chains.graph_qa.cypher import GraphCypherQAChain
from langchain_community.chat_models import ChatOpenAI, ChatOllama

# llm = ChatOpenAI(temperature=0, model='gpt-4o-mini')
llm = ChatOllama(temperature=0, model='llama3.1')

chain = GraphCypherQAChain.from_llm(
    cypher_llm=llm, qa_llm=llm, graph=graph, verbose=True, validate_cypher=True, qa_prompt=qa_generation_prompt, cypher_prompt=cypher_generation_prompt
)

In [4]:
# result = chain.invoke({"query": "Undang Undang No 2 Tahun 1964 mencabut peraturan apa saja?"})
# result = chain.invoke({"query": "Undang Undang No 36 Tahun 1953 dicabut peraturan apa saja?"})
# result = chain.invoke({"query": "Undang Undang No 14 Tahun 1957 mengubah peraturan apa saja?"})
result = chain.invoke({"query": "Undang Undang No 3 Tahun 1955 diubah peraturan apa saja?"})



> Entering new GraphCypherQAChain chain...
Generated Cypher:
cypher
MATCH (p:Peraturan {nomor: '3', tahun: '1955', jenis_peraturan: 'UNDANG-UNDANG'})
MATCH (p)-[:DIUBAH]->(revoked:Peraturan)
RETURN 
  p AS revokedRegulation, 
  revoked AS revokingRegulation

Full Context:
[{'revokedRegulation': {'tentang': 'PENGUBAHAN DAN TAMBAHAN ORDONANSI BEA-KELUAR-UMUM 1949)', 'tahun': '1955', 'jenis_peraturan': 'UNDANG-UNDANG', 'tempat_penetapan': 'Jakarta', 'id': 'https://peraturan.go.id/id/uu-no-3-tahun-1955', 'pemrakarsa': 'PEMERINTAH PUSAT', 'nomor': '3', 'status': 'Berlaku'}, 'revokingRegulation': {'tentang': 'PENGESAHAN UNDANG-UNDANG DARURAT NO. 3 TAHUN 1955 (LEMBARAN NEGARA TAHUN 1955 NO. 15) TENTANG PENUNJUKAN PELABUHAN PALEMBANG MENJADI PERUSAHAAN NEGARA DALAM ARTI "INDISCHE BEDRIJVENWET" (STAATSBLAD 1927 NO. 419) MENJADI UNDANG-UNDANG', 'tahun': '1960', 'jenis_peraturan': 'UNDANG-UNDANG', 'tempat_penetapan': 'Jakarta', 'id': 'https://peraturan.go.id/id/uu-no-3-tahun-1960', 'pemrakarsa'

In [5]:
print(result['result'])

Pertanyaannya adalah: Undang-Undang No 3 Tahun 1955 diubah peraturan apa saja?

Jawabannya adalah:

Undang-Undang No 3 Tahun 1955 telah diubah oleh Undang-Undang No 3 Tahun 1960. Perubahan ini ditetapkan dalam Lembaran Negara Tahun 1960 No 15, yang merupakan pengesahan dari Undang-Undang Darurat No 3 Tahun 1955.

Informasi lebih lanjut tentang peraturan yang diubah adalah:

* Nomor: 3
* Jenis Peraturan: Undang-Undang
* Tempat Penetapan: Jakarta
* Pemrakarsa: Pemerintah Pusat
* Tahun: 1960

Perlu diingat bahwa perubahan ini telah ditetapkan dalam Undang-Undang No 3 Tahun 1960, yang merupakan pengganti dari Undang-Undang No 3 Tahun 1955.
